## Self-Supervised Speech Recognition - Toronto Dataset

In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
from pathlib import Path
import pandas as pd
import torch
import librosa
from transformers import (
  AutoProcessor,
  Data2VecAudioForCTC,
  Wav2Vec2CTCTokenizer,
  Wav2Vec2FeatureExtractor,
  Wav2Vec2Processor,
)
from src.dataset import TorontoDataset, DataCollatorCTCWithPadding, sanitize
from torch.utils.data import DataLoader
import IPython.display as ipd
import lightning as L
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.callbacks import ModelCheckpoint
import evaluate
import os
import json
from sklearn.model_selection import GroupKFold

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
DATA_DIR = Path("data")
TORONTO = DATA_DIR / Path("toronto_clean")
TIMIT = DATA_DIR / Path("timit")

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"


## Toronto Dataset Speech Recognition

In [3]:
toronto_df = pd.read_parquet(TORONTO / "toronto.parquet")
toronto_train_df = toronto_df[toronto_df["split"] == "train"]
toronto_test_df = toronto_df[toronto_df["split"] == "test"]


In [4]:
BASELINE_MODEL = "Respeecher/ukrainian-data2vec-asr"

baseline_processor = AutoProcessor.from_pretrained(BASELINE_MODEL)
baseline_model = Data2VecAudioForCTC.from_pretrained(BASELINE_MODEL)


Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

In [5]:
baseline_model.eval()

sample = toronto_test_df.iloc[0]

audio, sr = librosa.load(sample.path, sr=16_000)

print(sample)

inputs = baseline_processor(audio, sampling_rate=sr, return_tensors="pt")

with torch.no_grad():
    logits = baseline_model(**inputs).logits

predicted_ids = torch.argmax(logits, dim=-1)

transcription = baseline_processor.batch_decode(predicted_ids)

print(f"Original:\t{sample.label}")
print(f"Transcript:\t{transcription[0]}")

ipd.Audio(sample.path)


filename                                      toronto_123_1.wav
path           data/toronto_clean/toronto_123/toronto_123_1.wav
speaker_id                                          toronto_123
duration                                                   6.76
label         валарморгулісне україна за три тижні до виборі...
split                                                      test
Name: 751, dtype: object
Original:	валарморгулісне україна за три тижні до виборів президента та за п’ять тижнів до нового сезону гри престолів
Transcript:	валоремургулєсне україна за три тижні до виборів призидента та за пять тижнів до нового сезону гри пристолів


In [6]:
collator = DataCollatorCTCWithPadding(processor=baseline_processor)

toronto_train_ds = TorontoDataset(toronto_train_df, baseline_processor, TORONTO, features_column="input_values")
toronto_val_ds = TorontoDataset(toronto_test_df, baseline_processor, TORONTO, features_column="input_values")

toronto_train_dl = DataLoader(toronto_train_ds, batch_size=8, shuffle=True, collate_fn=collator)
toronto_val_dl = DataLoader(toronto_val_ds, batch_size=8, shuffle=False, collate_fn=collator)


In [7]:
class Data2VecCTC(L.LightningModule):
  def __init__(self, model, processor):
    super().__init__()
    self.model = model
    self.processor = processor
    self.wer = evaluate.load("wer")
    self.cer = evaluate.load("cer")

  def on_train_epoch_start(self):
    self.model.train()

  def training_step(self, batch, _):
    loss = self.model(**batch).loss
    self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
    return loss

  def validation_step(self, batch, _):
    out = self.model(**batch)
    self.log("val_loss", out.loss, prog_bar=True)

    pred_ids = torch.argmax(out.logits, dim=-1)

    labels = batch["labels"].clone()
    labels[labels == -100] = self.processor.tokenizer.pad_token_id

    preds = self.processor.batch_decode(pred_ids)
    refs = self.processor.batch_decode(labels, group_tokens=False)

    self.wer.add_batch(predictions=preds, references=refs)
    self.cer.add_batch(predictions=preds, references=refs)

  def on_validation_epoch_end(self):
    self.log("val_wer", self.wer.compute(), prog_bar=True)
    self.log("val_cer", self.cer.compute(), prog_bar=True)

  def configure_optimizers(self):
    head_params = self.model.lm_head.parameters()
    encoder_params = [p for p in self.model.data2vec_audio.parameters() if p.requires_grad]
    return torch.optim.AdamW([
      {"params": head_params, "lr": 1e-4},
      {"params": encoder_params, "lr": 1e-5},
    ])


In [8]:
baseline_collator = DataCollatorCTCWithPadding(
    processor=baseline_processor,
)

toronto_val_dl = DataLoader(toronto_val_ds, batch_size=8, shuffle=False, collate_fn=baseline_collator)

logger = CSVLogger("logs", name="data2vec_baseline_val")
baseline = Data2VecCTC(baseline_model, baseline_processor)

trainer = L.Trainer(accelerator=device, logger=logger)
trainer.validate(baseline, dataloaders=toronto_val_dl)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
You are using a CUDA device ('NVIDIA GeForce RTX 4070 SUPER') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

/opt/miniforge3/envs/deep-audio/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/opt/miniforge3/envs/deep-audio/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│          val_cer          │    0.16432054340839386    │
│         val_loss          │    1.3721251487731934     │
│          val_wer          │    0.3944547772407532     │
└───────────────────────────┴───────────────────────────┘

[{'val_loss': 1.3721251487731934,
  'val_wer': 0.3944547772407532,
  'val_cer': 0.16432054340839386}]

In [8]:
VOCAB = TORONTO / "vocab.json"

all_text = " ".join(toronto_train_df["label"])
vocab = sorted(set(all_text))
vocab_dict = {v: i for i, v in enumerate(vocab)}
vocab_dict["|"] = vocab_dict.pop(" ")
vocab_dict["<unk>"] = len(vocab_dict)
vocab_dict["<pad>"] = len(vocab_dict)

with open(VOCAB, "w", encoding="utf-8") as f:
    json.dump(vocab_dict, f, ensure_ascii=False)


In [9]:
BACKBONE_MODEL = "Respeecher/ukrainian-data2vec"

tokenizer = Wav2Vec2CTCTokenizer(VOCAB)

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(BACKBONE_MODEL)
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

model = Data2VecAudioForCTC.from_pretrained(
  BACKBONE_MODEL,
  vocab_size=len(vocab_dict),
  pad_token_id=tokenizer.pad_token_id,
  ctc_zero_infinity=True,
  ctc_loss_reduction="mean",
  attn_implementation="sdpa",
)

model.freeze_feature_encoder()


Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

[transformers] Data2VecAudioForCTC LOAD REPORT from: Respeecher/ukrainian-data2vec
Key            | Status  | 
---------------+---------+-
lm_head.weight | MISSING | 
lm_head.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
collator = DataCollatorCTCWithPadding(processor=processor)

gkf = GroupKFold(n_splits=5)
train_idx, val_idx = next(gkf.split(toronto_train_df, groups=toronto_train_df["speaker_id"]))

toronto_tr_df  = toronto_train_df.iloc[train_idx]
toronto_val_df = toronto_train_df.iloc[val_idx]

toronto_train_ds = TorontoDataset(toronto_tr_df, processor, TORONTO, features_column="input_values")
toronto_val_ds = TorontoDataset(toronto_val_df, processor, TORONTO, features_column="input_values")
toronto_test_ds = TorontoDataset(toronto_test_df, processor, TORONTO, features_column="input_values")

toronto_train_dl = DataLoader(
  toronto_train_ds,
  batch_size=8,
  shuffle=True,
  collate_fn=collator,
  num_workers=11,
  pin_memory=True,
  persistent_workers=True,
)

toronto_val_dl = DataLoader(
  toronto_val_ds,
  batch_size=8,
  shuffle=False,
  collate_fn=collator,
  num_workers=11,
  pin_memory=True,
)

toronto_test_dl = DataLoader(
  toronto_test_ds,
  batch_size=8,
  shuffle=False,
  collate_fn=collator,
  num_workers=11,
  pin_memory=True,
)

logger = CSVLogger("logs", name="toronto_c_data2vec_finetune_val_0_layers")
m = Data2VecCTC(model, processor)

checkpoint = ModelCheckpoint(
  monitor="val_wer",
  mode="min",
  save_top_k=1,
  filename="{epoch}-{val_wer:.3f}",
)

trainer = L.Trainer(
  accelerator=device,
  logger=logger,
  max_epochs=5,
  precision="bf16-mixed",
  callbacks=[checkpoint],
  gradient_clip_val=1.0,
)

model.train()

for param in model.data2vec_audio.parameters():
    param.requires_grad = False

trainer.fit(m, train_dataloaders=toronto_train_dl, val_dataloaders=toronto_val_dl)
model.eval()
trainer.validate(m, dataloaders=toronto_test_dl, ckpt_path="best")


Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA GeForce RTX 4070 SUPER') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/opt/miniforge3/envs/deep-audio/lib/python3.11/site-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision bf16-mixed is not supported by the model summary.  Estimated model size in MB will not be accura

┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type                ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ Data2VecAudioForCTC │  313 M │ train │     0 │
└───┴───────┴─────────────────────┴────────┴───────┴───────┘

Trainable params: 39.0 K                                                                                           
Non-trainable params: 313 M                                                                                        
Total params: 313 M                                                                                                
Total estimated model params size (MB): 1,253.261                                                                  
Modules in train mode: 429                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/opt/miniforge3/envs/deep-audio/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: 
`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

In [ ]:
model.eval()

with torch.no_grad():
    logits = model(**inputs).logits

predicted_ids = torch.argmax(logits, dim=-1)

transcription = processor.batch_decode(predicted_ids)

print(f"Original:\t{sample.label}")
print(f"Transcript:\t{transcription[0]}")


Original:	валарморгулісне україна за три тижні до виборів президента та за п’ять тижнів до нового сезону гри престолів
Transcript:	валрмуору гул яесне укроана за трі ттіжні до вибоіув прездеант та за пяаь тажнів до новог созону гри престполів


In [ ]:
print(predicted_ids[0][:60])
print("unique predicted ids:", predicted_ids.unique())
print("blank/pad id:", processor.tokenizer.pad_token_id)


tensor([37, 37, 37, 37, 37, 37, 37, 37, 37, 37,  4,  4,  2,  2,  2,  2, 37, 13,
        13, 13, 13, 13, 37, 37, 18, 18, 18, 18, 37, 14, 14, 14, 14, 21, 16, 37,
        18, 18, 18, 21, 37,  0,  5,  5,  5, 21, 21, 21, 37, 13,  0,  0,  0, 30,
         7, 19, 19, 19, 19, 19])
unique predicted ids: tensor([ 0,  2,  3,  4,  5,  6,  7,  8,  9, 10, 12, 13, 14, 15, 16, 17, 18, 19,
        20, 21, 28, 30, 32, 37])
blank/pad id: 37


In [ ]:
model = Data2VecAudioForCTC.from_pretrained(
  BACKBONE_MODEL,
  vocab_size=len(vocab_dict),
  pad_token_id=tokenizer.pad_token_id,
  ctc_zero_infinity=True,
  ctc_loss_reduction="mean",
  attn_implementation="sdpa",
)

model.freeze_feature_encoder()

collator = DataCollatorCTCWithPadding(processor=processor)

gkf = GroupKFold(n_splits=5)
train_idx, val_idx = next(gkf.split(toronto_train_df, groups=toronto_train_df["speaker_id"]))

toronto_tr_df  = toronto_train_df.iloc[train_idx]
toronto_val_df = toronto_train_df.iloc[val_idx]

toronto_train_ds = TorontoDataset(toronto_tr_df, processor, TORONTO, features_column="input_values")
toronto_val_ds = TorontoDataset(toronto_val_df, processor, TORONTO, features_column="input_values")
toronto_test_ds = TorontoDataset(toronto_test_df, processor, TORONTO, features_column="input_values")

toronto_train_dl = DataLoader(
  toronto_train_ds,
  batch_size=8,
  shuffle=True,
  collate_fn=collator,
  num_workers=11,
  pin_memory=True,
  persistent_workers=True,
)

toronto_val_dl = DataLoader(
  toronto_val_ds,
  batch_size=8,
  shuffle=False,
  collate_fn=collator,
  num_workers=11,
  pin_memory=True,
)

toronto_test_dl = DataLoader(
  toronto_test_ds,
  batch_size=8,
  shuffle=False,
  collate_fn=collator,
  num_workers=11,
  pin_memory=True,
)

logger = CSVLogger("logs", name="toronto_c_data2vec_finetune_val_4_layers")
m = Data2VecCTC(model, processor)

checkpoint = ModelCheckpoint(
  monitor="val_wer",
  mode="min",
  save_top_k=1,
  filename="{epoch}-{val_wer:.3f}",
)

trainer = L.Trainer(
  accelerator=device,
  logger=logger,
  max_epochs=5,
  precision="bf16-mixed",
  callbacks=[checkpoint],
  gradient_clip_val=1.0,
)

model.train()

for param in model.data2vec_audio.parameters():
  param.requires_grad = False


n_unfrozen = 4
for layer in model.data2vec_audio.encoder.layers[-n_unfrozen:]:
  for param in layer.parameters():
    param.requires_grad = True

trainer.fit(m, train_dataloaders=toronto_train_dl, val_dataloaders=toronto_val_dl)
model.eval()
trainer.validate(m, dataloaders=toronto_test_dl, ckpt_path="best")


Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

[transformers] Data2VecAudioForCTC LOAD REPORT from: Respeecher/ukrainian-data2vec
Key            | Status  | 
---------------+---------+-
lm_head.bias   | MISSING | 
lm_head.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/opt/miniforge3/envs/deep-audio/lib/python3.11/site-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision bf16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type                ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ Data2VecAudioForCTC │  313 M │ train │     0 │
└───┴───────┴─────────────────────┴────────┴───────┴───────┘

Trainable params: 50.4 M                                                                                           
Non-trainable params: 262 M                                                                                        
Total params: 313 M                                                                                                
Total estimated model params size (MB): 1,253.261                                                                  
Modules in train mode: 429                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

...

`Trainer.fit` stopped: `max_epochs=5` reached.


Restoring states from the checkpoint path at logs/toronto_data2vec_finetune_val_4_layers/version_0/checkpoints/epoch=4-val_wer=0.361.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at logs/toronto_data2vec_finetune_val_4_layers/version_0/checkpoints/epoch=4-val_wer=0.361.ckpt


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│          val_cer          │    0.16453856229782104    │
│         val_loss          │    0.9759981036186218     │
│          val_wer          │     0.384656697511673     │
└───────────────────────────┴───────────────────────────┘

[{'val_loss': 0.9759981036186218,
  'val_wer': 0.384656697511673,
  'val_cer': 0.16453856229782104}]

In [ ]:
model = Data2VecAudioForCTC.from_pretrained(
  BACKBONE_MODEL,
  vocab_size=len(vocab_dict),
  pad_token_id=tokenizer.pad_token_id,
  ctc_zero_infinity=True,
  ctc_loss_reduction="mean",
  attn_implementation="sdpa",
)

model.freeze_feature_encoder()

collator = DataCollatorCTCWithPadding(processor=processor)

gkf = GroupKFold(n_splits=5)
train_idx, val_idx = next(gkf.split(toronto_train_df, groups=toronto_train_df["speaker_id"]))

toronto_tr_df  = toronto_train_df.iloc[train_idx]
toronto_val_df = toronto_train_df.iloc[val_idx]

toronto_train_ds = TorontoDataset(toronto_tr_df, processor, TORONTO, features_column="input_values")
toronto_val_ds = TorontoDataset(toronto_val_df, processor, TORONTO, features_column="input_values")
toronto_test_ds = TorontoDataset(toronto_test_df, processor, TORONTO, features_column="input_values")

toronto_train_dl = DataLoader(
  toronto_train_ds,
  batch_size=8,
  shuffle=True,
  collate_fn=collator,
  num_workers=11,
  pin_memory=True,
  persistent_workers=True,
)

toronto_val_dl = DataLoader(
  toronto_val_ds,
  batch_size=8,
  shuffle=False,
  collate_fn=collator,
  num_workers=11,
  pin_memory=True,
)

toronto_test_dl = DataLoader(
  toronto_test_ds,
  batch_size=8,
  shuffle=False,
  collate_fn=collator,
  num_workers=11,
  pin_memory=True,
)

logger = CSVLogger("logs", name="toronto_c_data2vec_finetune_val_12_layers")
m = Data2VecCTC(model, processor)

checkpoint = ModelCheckpoint(
  monitor="val_wer",
  mode="min",
  save_top_k=1,
  filename="{epoch}-{val_wer:.3f}",
)

trainer = L.Trainer(
  accelerator=device,
  logger=logger,
  max_epochs=5,
  precision="bf16-mixed",
  callbacks=[checkpoint],
  gradient_clip_val=1.0,
)

model.train()

for param in model.data2vec_audio.parameters():
  param.requires_grad = False


n_unfrozen = len(model.data2vec_audio.encoder.layers) // 2
for layer in model.data2vec_audio.encoder.layers[-n_unfrozen:]:
  for param in layer.parameters():
    param.requires_grad = True

trainer.fit(m, train_dataloaders=toronto_train_dl, val_dataloaders=toronto_val_dl)
model.eval()
trainer.validate(m, dataloaders=toronto_test_dl, ckpt_path="best")


Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

[transformers] Data2VecAudioForCTC LOAD REPORT from: Respeecher/ukrainian-data2vec
Key            | Status  | 
---------------+---------+-
lm_head.bias   | MISSING | 
lm_head.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/opt/miniforge3/envs/deep-audio/lib/python3.11/site-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision bf16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type                ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ Data2VecAudioForCTC │  313 M │ train │     0 │
└───┴───────┴─────────────────────┴────────┴───────┴───────┘

Trainable params: 151 M                                                                                            
Non-trainable params: 162 M                                                                                        
Total params: 313 M                                                                                                
Total estimated model params size (MB): 1,253.261                                                                  
Modules in train mode: 429                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

...

`Trainer.fit` stopped: `max_epochs=5` reached.


Restoring states from the checkpoint path at logs/toronto_data2vec_finetune_val_12_layers/version_0/checkpoints/epoch=4-val_wer=0.285.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at logs/toronto_data2vec_finetune_val_12_layers/version_0/checkpoints/epoch=4-val_wer=0.285.ckpt


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│          val_cer          │    0.13473504781723022    │
│         val_loss          │    0.9038810729980469     │
│          val_wer          │    0.3125036060810089     │
└───────────────────────────┴───────────────────────────┘

[{'val_loss': 0.9038810729980469,
  'val_wer': 0.3125036060810089,
  'val_cer': 0.13473504781723022}]

In [ ]:
model = Data2VecAudioForCTC.from_pretrained(
  BACKBONE_MODEL,
  vocab_size=len(vocab_dict),
  pad_token_id=tokenizer.pad_token_id,
  ctc_zero_infinity=True,
  ctc_loss_reduction="mean",
  attn_implementation="sdpa",
)

model.freeze_feature_encoder()

collator = DataCollatorCTCWithPadding(processor=processor)

gkf = GroupKFold(n_splits=5)
train_idx, val_idx = next(gkf.split(toronto_train_df, groups=toronto_train_df["speaker_id"]))

toronto_tr_df  = toronto_train_df.iloc[train_idx]
toronto_val_df = toronto_train_df.iloc[val_idx]

toronto_train_ds = TorontoDataset(toronto_tr_df, processor, TORONTO, features_column="input_values")
toronto_val_ds = TorontoDataset(toronto_val_df, processor, TORONTO, features_column="input_values")
toronto_test_ds = TorontoDataset(toronto_test_df, processor, TORONTO, features_column="input_values")

toronto_train_dl = DataLoader(
  toronto_train_ds,
  batch_size=8,
  shuffle=True,
  collate_fn=collator,
  num_workers=11,
  pin_memory=True,
  persistent_workers=True,
)

toronto_val_dl = DataLoader(
  toronto_val_ds,
  batch_size=8,
  shuffle=False,
  collate_fn=collator,
  num_workers=11,
  pin_memory=True,
)

toronto_test_dl = DataLoader(
  toronto_test_ds,
  batch_size=8,
  shuffle=False,
  collate_fn=collator,
  num_workers=11,
  pin_memory=True,
)

logger = CSVLogger("logs", name="toronto_c_data2vec_finetune_val_24_layers")
m = Data2VecCTC(model, processor)

checkpoint = ModelCheckpoint(
  monitor="val_wer",
  mode="min",
  save_top_k=1,
  filename="{epoch}-{val_wer:.3f}",
)

trainer = L.Trainer(
  accelerator=device,
  logger=logger,
  max_epochs=5,
  precision="bf16-mixed",
  callbacks=[checkpoint],
  gradient_clip_val=1.0,
)

model.train()

for param in model.data2vec_audio.parameters():
  param.requires_grad = False


for layer in model.data2vec_audio.encoder.layers:
  for param in layer.parameters():
    param.requires_grad = True

trainer.fit(m, train_dataloaders=toronto_train_dl, val_dataloaders=toronto_val_dl)
model.eval()
trainer.validate(m, dataloaders=toronto_test_dl, ckpt_path="best")


Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

[transformers] Data2VecAudioForCTC LOAD REPORT from: Respeecher/ukrainian-data2vec
Key            | Status  | 
---------------+---------+-
lm_head.bias   | MISSING | 
lm_head.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/opt/miniforge3/envs/deep-audio/lib/python3.11/site-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision bf16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type                ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ Data2VecAudioForCTC │  313 M │ train │     0 │
└───┴───────┴─────────────────────┴────────┴───────┴───────┘

Trainable params: 302 M                                                                                            
Non-trainable params: 11.0 M                                                                                       
Total params: 313 M                                                                                                
Total estimated model params size (MB): 1,253.261                                                                  
Modules in train mode: 429                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

...

`Trainer.fit` stopped: `max_epochs=5` reached.


Restoring states from the checkpoint path at logs/toronto_data2vec_finetune_val_12_layers/version_1/checkpoints/epoch=4-val_wer=0.222.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at logs/toronto_data2vec_finetune_val_12_layers/version_1/checkpoints/epoch=4-val_wer=0.222.ckpt


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│          val_cer          │    0.10652419179677963    │
│         val_loss          │    0.8179080486297607     │
│          val_wer          │    0.2542135715484619     │
└───────────────────────────┴───────────────────────────┘

[{'val_loss': 0.8179080486297607,
  'val_wer': 0.2542135715484619,
  'val_cer': 0.10652419179677963}]

In [ ]:
sample = toronto_test_df.iloc[300]

audio, sr = librosa.load(sample.path, sr=16_000)

inputs = baseline_processor(audio, sampling_rate=sr, return_tensors="pt")

print(sample)

with torch.no_grad():
    logits = baseline_model(**inputs).logits

predicted_ids = torch.argmax(logits, dim=-1)

baseline_transcription = baseline_processor.batch_decode(predicted_ids)

inputs = processor(audio, sampling_rate=sr, return_tensors="pt")

print(sample)

with torch.no_grad():
    logits = model(**inputs).logits

predicted_ids = torch.argmax(logits, dim=-1)

transcription = processor.batch_decode(predicted_ids)

print(f"Original:\t{sample.label}")
print(f"Baseline:\t{baseline_transcription[0]}")
print(f"Fine-tuned:\t{transcription[0]}")


filename                                     toronto_134_41.wav
path                data/toronto/toronto_134/toronto_134_41.wav
speaker_id                                          toronto_134
duration                                                    6.0
label         потрапляння в топ рейтингу порнхабу дає надію ...
Name: 2423, dtype: object
filename                                     toronto_134_41.wav
path                data/toronto/toronto_134/toronto_134_41.wav
speaker_id                                          toronto_134
duration                                                    6.0
label         потрапляння в топ рейтингу порнхабу дає надію ...
Name: 2423, dtype: object
Original:	потрапляння в топ рейтингу порнхабу дає надію що світові порнографічні продакшени почнуть нарешті випускати
Baseline:	потрапляння в стол прейтимгу хортхабу дає надії що світові порнографічні продакшини почнуть нарешті випускати
Fine-tuned:	потрапляння втоп рейтингу корнхабу дає надії що світові по